### クラスリングと確率の計算

選択した説明変数でクラスタリングを行い原子環境の確率を計算する。


In [ ]:
import pandas as pd
g_df = pd.read_csv(
    "../data_calculated/Carbon8_descriptor.csv", index_col=[0, 1])
g_df


In [ ]:
from itertools import combinations
import sklearn.mixture
import collections
import sklearn.cluster
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
%matplotlib inline

"""
3. data analysis
"""


def gmm2D_predict(X_, n_components, random_state):
    """ plot clusters in the descriptor space

    Args:
        X_ (np.array): descriptor
        n_components (int): the number of clusters
        random_state (int): random state for GMM clustering
        title (str, optional): title of the figure. Defaults to None.
        filename (str, optional): filename. Defaults to None.

    Returns:
        clf: classification
        collections.Counter: the number of samples in clusters
    """
    X = X_.copy()
    colors = ["b", "r", "y", "m", "c"]
    fig, axes = plt.subplots(1, 2)
    ax = axes[0]
    sns.kdeplot(x=X[:, 0], y=X[:, 1], ax=ax)

    ax = axes[1]
    sns.kdeplot(x=X[:, 0], y=X[:, 1], fill=True, ax=ax)

    clf = sklearn.mixture.GaussianMixture(
        n_components=n_components, random_state=random_state)
    clf.fit(X)
    print("means", clf.means_)
    print("covariance", clf.covariances_)
    yp = clf.predict(X)
    count = collections.Counter(yp)

    for i in range(n_components):
        data = []
        for x1, y1 in zip(X_, yp):
            if y1 == i:
                data.append(x1)
        data = np.array(data)
        if data.shape[0] > 0:
            ax.plot(data[:, 0], data[:, 1], colors[i]+".", label=str(i))

    ax.plot(clf.means_[:, 0], clf.means_[:, 1], "o")
    ax.legend()

    return clf, count


"""
4. visualiztion
"""


def make_gmm2D_plot2(X_, pairs, n_components, random_state):
    """helper routine for gmm2D_predict

    Args:
        X_ (np.array): descriptor
        pairs (list): a list of feature pairs
        n_components (int): the number of clusters
        random_state (int): random state
    """
    for i, j in pairs:
        print("pairs", i, j)
        vallist = []
        for ncl in n_components:
            print("cluster", ncl)
            clf, count = gmm2D_predict(X_[:, [i, j]], ncl, random_state)
            vallist.append([i, j, ncl, count])
        for iv, x in enumerate(vallist):
            if iv == 0:
                print(iv == 0, x)
            else:
                print(iv == 0, x, vallist[iv][3]-vallist[iv-1][3])


def try_gmm(df):
    """plot feature pairs

    Args:
        df (pd.DataFrame): data
    """
    X = df.values
    labels = df.columns
    pairs = [[5, 10]]  # ２つ使用する説明変数の定義
    for i in range(1):
        make_gmm2D_plot2(X, pairs=pairs, n_components=[3], random_state=i)


try_gmm(g_df)


In [ ]:
def try_gmm(df):
    """plot feature pairs

    Args:
        df (pd.DataFrame): data
    """
    X = df.values
    labels = df.columns
    pairs = [[4, 6]]  # ２つ使用する説明変数の定義
    for i in range(1):
        make_gmm2D_plot2(X, pairs, [3, 4], i)


try_gmm(g_df)


GMM線形回帰で使用するclusteringをおこなう。

In [ ]:

import pickle


def gmm2D_plot(X, yp, clf, filename=None):
    """plot descriptors in 2D

    Args:
        X (np.array): descriptor
        yp (np.array): target values
        clf (classifier): classifier
        filename (str, optional): filename. Defaults to None.

    Returns:
        Counter: the numbers in the cluster classes
    """
    n_components = clf.n_components
    fig, axes = plt.subplots(1, 2)
    colors = ["b", "r", "y", "m", "c"]
    ax = axes[0]
    sns.kdeplot(x=X[:, 0], y=X[:, 1], ax=ax)

    ax = axes[1]
    sns.kdeplot(x=X[:, 0], y=X[:, 1], fill=True, ax=ax)

    count = collections.Counter(yp)

    for i in range(n_components):
        data = []
        for x1, y1 in zip(X, yp):
            if y1 == i:
                data.append(x1)
        data = np.array(data)
        ax.plot(data[:, 0], data[:, 1], colors[i]+".", label=str(i))

    ax.plot(clf.means_[:, 0], clf.means_[:, 1], "o")
    ax.legend()
    return count


def gmm2D_predict(X_, n_components, random_state):
    """predict and plot the clustering

    Args:
        X_ (np.array): descriptor
        n_components (int): the number of clusters
        random_state (int): randum state

    Returns:
        classifier: classifier
        Counter: the number of samples in clusters
    """

    X = X_.copy()

    # also set initial positions to make the result the same
    means_init = [[0.02136692, 1.56231726],
                  [0.03099394, 1.87859832],
                  [0.01374351, 1.04265199],
                  [0.01585623, 0.58043088]]
    means_init = np.array(means_init)

    clf = sklearn.mixture.GaussianMixture(
        n_components=n_components, means_init=means_init, random_state=random_state)

    clf.fit(X)
    print("means", clf.means_)
    print("covariance", clf.covariances_)
    yp = clf.predict(X)

    count = gmm2D_plot(X, yp, clf)

    return clf, count


def make_gmm2D(X_, pair, ncluster, random_state):
    """helper subroutine of gmm2D_predict

    Args:
        X_ (np.array): descriptor
        pair (list): a list of feature pairs
        ncluster (int): a list of the number of clusters
        random_state (int): random state

    Returns:
        classifier: GMM
        Counter: the number of samples in the clusters
    """
    for i, j in pair:
        vallist = []
        for ncl in ncluster:
            print("cluster", ncl)
            clf, count = gmm2D_predict(X_[:, [i, j]], ncl, random_state)
            vallist.append([i, j, ncl, count])

        # error check
        for iv, x in enumerate(vallist):
            if iv == 0:
                print(iv == 0, x)
            else:
                print(iv == 0, x, vallist[iv][3]-vallist[iv-1][3])
    return clf, count


def try_gmm(df):
    """plot feature pairs

    Args:
        df (pd.DataFrame): data
    """
    X = df.values
    labels = df.columns
    pairs = [[4, 6]]
    for i in range(2, 3):
        clf, count = make_gmm2D(X, pairs, [4], i)
    return clf


g_clf = try_gmm(g_df)

import os
os.makedirs("models", exist_ok=True)
with open("models/gmm.pickle", "wb") as _f:
    pickle.dump(g_clf, _f)
print("done")


make dataframe to save it

In [ ]:
def make_probalabel(n_components):
    """make label which contains cluster + number 

    Args:
        n_components (int): the number of clusters

    Returns:
        list: a list of label
    """
    labels = []
    for ic in range(n_components):
        labels.append("cluster"+str(ic))
    return labels


def make_df_cluster(df, pair, clf):
    """make clusters

    Args:
        df (pd.DataFrame): descriptor
        pair (list): a list of descriptor pairs
        clf (classifier): classifier

    Returns:
        pd.DataFrame: predicted y , DataFrame of predicted probability 
    """

    n_components = clf.n_components

    X = df.values

    print("X.shape", X.shape)

    yp = clf.predict(X[:, pair])
    print("yp.shape", yp.shape)
    print("yp", yp)

    count = collections.Counter(yp)

    print("count", count)

    yproba = clf.predict_proba(X[:, pair])
    print("yproba.shape", yproba.shape)
    print(yproba[0, :], yp[0])
    print(yproba[1, :], yp[1])

    proba_labels = ["cluster"+str(i) for i in range(n_components)]
    df_yproba = pd.DataFrame(yproba, columns=proba_labels, index=df.index)

    df_yp = pd.DataFrame(yp.reshape(yp.shape[0], 1), columns=[
                         "y_predict"], index=df.index)

    return df_yp, df_yproba


print("df.shape", g_df.shape)
g_df_yp, g_df_yproba = make_df_cluster(g_df, [4, 6], g_clf)


クラスタラベルの表示

In [ ]:
g_df_yp


各クラスタ確率の表示

In [ ]:
g_df_yproba


セーブを行う。

In [ ]:
g_df_yproba.to_csv("../data_calculated/Carbon8_yproba.csv")
g_df_yp.to_csv("../data_calculated/Carbon8_yp.csv")


In [ ]:
print("all done")
